In [21]:
# Step 1: Imports
import os, glob
from sentence_transformers import SentenceTransformer
import chromadb
from typing import TypedDict



In [22]:
# Step 2: Ingest docs and build embeddings

model = SentenceTransformer("all-MiniLM-L6-v2")
client = chromadb.PersistentClient(path="chroma_db")
collection = client.get_or_create_collection("zepto_policies")

# Embed all docs
for file in glob.glob("docs/*.txt"):
    with open(file, "r", encoding="utf-8") as f:
        text = f.read()
        embedding = model.encode(text).tolist()
        collection.add(documents=[text], embeddings=[embedding], ids=[os.path.basename(file)])

print("✅ Corpus embedded and stored in ChromaDB")



Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5334.68it/s]


✅ Corpus embedded and stored in ChromaDB


In [24]:
# Step 3: Define LangGraph state and intent classifier

MOCK_LLM = os.getenv("MOCK_LLM", "1")

class State(TypedDict):
    query: str
    intent: str
    answer: str
    sources: list[str]
    confidence: float

def classify_intent(state: State) -> State:
    q = state["query"].lower()
    keywords = ["delivery","return","refund","membership","tracking","cancel","gift card","support hours"]
    state["intent"] = "policy_question" if any(k in q for k in keywords) else "general_question"
    print("Intent classified as:", state["intent"])
    return state



In [25]:
# Step 4: Retrieval node with safe indexing

def retrieve_and_answer(state: State) -> State:
    results = collection.query(query_texts=[state["query"]], n_results=3)

    # Defensive check to avoid IndexError
    if results["documents"] and results["documents"][0]:
        top_chunk = results["documents"][0][0][:200]
        if MOCK_LLM == "1":
            state["answer"] = f"Based on the retrieved context: {top_chunk}"
        else:
            state["answer"] = "(LLM grounded answer)"
        state["sources"] = results["ids"][0]
    else:
        state["answer"] = "No relevant context found in corpus."
        state["sources"] = []

    state["confidence"] = 1.0
    print("Retrieved sources:", state["sources"])
    return state


In [26]:
# Step 5: Direct answer node

def direct_answer(state: State) -> State:
    if MOCK_LLM == "1":
        state["answer"] = "I can only answer questions about Zepto policies right now."
    else:
        state["answer"] = "(LLM direct answer)"
    state["sources"] = []
    state["confidence"] = 1.0
    print("Direct answer returned.")
    return state



In [27]:
# Step 6: Example runs in mock mode

# Example 1: Policy question
state = {"query": "What is Zepto's delivery policy?", "intent": "", "answer": "", "sources": [], "confidence": 0.0}
state = classify_intent(state)
state = retrieve_and_answer(state)
print(state)

# Example 2: General question
state = {"query": "Who is the CEO of Zepto?", "intent": "", "answer": "", "sources": [], "confidence": 0.0}
state = classify_intent(state)
state = direct_answer(state)
print(state)


Intent classified as: policy_question
Retrieved sources: []
{'query': "What is Zepto's delivery policy?", 'intent': 'policy_question', 'answer': 'No relevant context found in corpus.', 'sources': [], 'confidence': 1.0}
Intent classified as: general_question
Direct answer returned.
{'query': 'Who is the CEO of Zepto?', 'intent': 'general_question', 'answer': 'I can only answer questions about Zepto policies right now.', 'sources': [], 'confidence': 1.0}
